In [ ]:
import os
import re
import glob
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np

# PDF text extraction
import pdfplumber

# Similarity + clustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# PDF encryption / permissions
try:
    from PyPDF2 import PdfReader  # pip: PyPDF2
except ImportError as e:
    raise ImportError("PyPDF2 is required for password/permission detection. Install via `pip install PyPDF2`.") from e


# ----------------------------
# Config you can tweak
# ----------------------------
DEFAULT_SIM_THRESHOLD = 0.78   # higher = stricter grouping (try 0.75–0.88)
MAX_PAGES_TO_SAMPLE = 1        # templates usually show up on page 1
MAX_LINES = 80                 # take only first N lines (stable header/table labels)
DROP_NUMBER_HEAVY_LINES = True # drop lines that are mostly numbers (NAV values etc.)


@dataclass
class PdfTemplateFingerprint:
    path: str
    fingerprint_text: str


def _normalize_line(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)

    # Remove things that vary per statement
    s = re.sub(r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b", " <DATE> ", s)  # 01-02-2026
    s = re.sub(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b", " <DATE> ", s)    # 2026-03-02
    s = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " <EMAIL> ", s, flags=re.I)

    # Replace long numbers / currency-like values (NAV, amounts, folio, etc.)
    s = re.sub(r"₹", " INR ", s)
    s = re.sub(r"\bINR\b", " INR ", s, flags=re.I)
    s = re.sub(r"\b\d{6,}\b", " <LONGNUM> ", s)     # long ids
    s = re.sub(r"\b\d+\.\d+\b", " <DECIMAL> ", s)   # decimals
    s = re.sub(r"\b\d+\b", " <NUM> ", s)            # remaining integers

    # Keep letters and common punctuation for template cues
    s = re.sub(r"[^a-zA-Z<>/ _:-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _is_number_heavy(line: str) -> bool:
    raw = re.sub(r"\s+", "", line)
    if not raw:
        return True
    digits = sum(ch.isdigit() for ch in raw)
    return digits / max(1, len(raw)) > 0.45


def inspect_pdf_security(pdf_path: str) -> Tuple[bool, bool]:
    """
    Returns:
      - is_password_protected: True if encrypted and cannot be opened with empty password
      - has_copy_permission: for encrypted PDFs, inferred from /Encrypt /P bitmask (copy flag);
                             for unencrypted PDFs, True.
    Notes:
      - If password protected (can't decrypt), copy permission is treated as False (unknown in practice).
    """
    try:
        reader = PdfReader(pdf_path)

        if not reader.is_encrypted:
            return False, True  # not protected; no permission restrictions

        # Encrypted: try decrypt with empty password (common for "restrictions only" PDFs)
        try:
            decrypt_result = reader.decrypt("")  # returns 0 if failed, 1/2 if success (PyPDF2 behavior)
        except Exception:
            decrypt_result = 0

        if decrypt_result == 0:
            # Can't decrypt without a password
            return True, False

        # Decrypted successfully: check permissions from /Encrypt dictionary
        try:
            encrypt_dict = reader.trailer.get("/Encrypt", None)
            if encrypt_dict and "/P" in encrypt_dict:
                p_val = int(encrypt_dict["/P"])
                # PDF permission bit for "copy/extract" is 0x10 (16)
                has_copy = bool(p_val & 16)
                return False, has_copy
        except Exception:
            pass

        # If we can't read permission flags but decrypted, assume copy allowed (conservative alternative: False)
        return False, True

    except Exception:
        # If reader can't open at all, treat as password protected-ish / inaccessible
        return True, False


def fingerprint_pdf(pdf_path: str,
                    max_pages: int = MAX_PAGES_TO_SAMPLE,
                    max_lines: int = MAX_LINES) -> PdfTemplateFingerprint:
    """
    Create a template fingerprint string for a PDF:
    - Extract page text (usually page 1 is enough)
    - Keep early lines (header, column labels, section titles)
    - Normalize away dates/numbers (variable content)
    """
    lines_out: List[str] = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages[:max_pages]
            for p in pages:
                txt = p.extract_text() or ""
                raw_lines = txt.splitlines()[:max_lines]
                for ln in raw_lines:
                    if DROP_NUMBER_HEAVY_LINES and _is_number_heavy(ln):
                        continue
                    norm = _normalize_line(ln)
                    if norm and len(norm) >= 3:
                        lines_out.append(norm)

        # De-dup while preserving order
        seen = set()
        uniq = []
        for l in lines_out:
            if l not in seen:
                uniq.append(l)
                seen.add(l)

        fp_text = "\n".join(uniq)
        return PdfTemplateFingerprint(path=pdf_path, fingerprint_text=fp_text)

    except Exception as e:
        # If a PDF is scanned-only or unreadable, fingerprint may be empty/error-tagged
        return PdfTemplateFingerprint(path=pdf_path, fingerprint_text=f"__ERROR__ {e}")


def cluster_by_similarity(fps: List[PdfTemplateFingerprint],
                          sim_threshold: float = DEFAULT_SIM_THRESHOLD
                          ) -> Tuple[List[int], np.ndarray]:
    """
    Clustering via similarity graph:
    - Compute cosine similarity matrix
    - Make edges where sim >= threshold
    - Connected components = clusters
    """
    docs = [fp.fingerprint_text for fp in fps]
    vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
    X = vec.fit_transform(docs)

    sim = cosine_similarity(X)
    n = sim.shape[0]

    # Build adjacency list
    adj: Dict[int, List[int]] = {i: [] for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= sim_threshold:
                adj[i].append(j)
                adj[j].append(i)

    # Connected components
    labels = [-1] * n
    cluster_id = 0
    for i in range(n):
        if labels[i] != -1:
            continue
        stack = [i]
        labels[i] = cluster_id
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if labels[v] == -1:
                    labels[v] = cluster_id
                    stack.append(v)
        cluster_id += 1

    return labels, sim


def classify_folder_to_csv(pdf_folder: str,
                           output_csv_path: str,
                           pattern: str = "*.pdf",
                           sim_threshold: float = DEFAULT_SIM_THRESHOLD) -> str:
    """
    Scans a folder, clusters PDFs, inspects security flags, and writes a CSV with:
      folder, filename, cluster, is_password_protected, has_copy_permission
    Returns output_csv_path.
    """
    pdf_paths = sorted(glob.glob(os.path.join(pdf_folder, pattern)))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in: {pdf_folder}")

    # Build fingerprints (template cues)
    fps = [fingerprint_pdf(p) for p in pdf_paths]

    # Cluster by similarity
    labels, _sim = cluster_by_similarity(fps, sim_threshold=sim_threshold)

    # Build CSV rows
    rows = []
    for fp, cluster in zip(fps, labels):
        folder = os.path.dirname(fp.path)
        filename = os.path.basename(fp.path)

        is_pw, has_copy = inspect_pdf_security(fp.path)

        rows.append((folder, filename, cluster, is_pw, has_copy))

    # Write CSV
    import csv
    os.makedirs(os.path.dirname(output_csv_path) or ".", exist_ok=True)
    with open(output_csv_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["folder", "filename", "cluster", "is_password_protected", "has_copy_permission"])
        w.writerows(rows)

    # Console summary
    print(f"Wrote: {output_csv_path}")
    print(f"Total PDFs: {len(pdf_paths)} | clusters: {len(set(labels))} | threshold={sim_threshold}")
    return output_csv_path


if __name__ == "__main__":
    # Example:
    # python classify_cas_templates.py
    # (set env vars or edit below)
    folder = os.environ.get("CAS_PDF_FOLDER", "")
    out_csv = os.environ.get("CAS_OUT_CSV", "cas_template_clusters.csv")

    if not folder:
        print("Set CAS_PDF_FOLDER env var to your PDF directory.")
    else:
        classify_folder_to_csv(folder, out_csv, sim_threshold=DEFAULT_SIM_THRESHOLD)